In [1]:
from pathlib import Path
raw_folder = Path(
    r"C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection"
    r"\data\raw\paderborn"
)
print("folder exists:", raw_folder.exists())

folder exists: True


In [2]:
#find the mat files
files  =sorted(raw_folder.glob("**/*.mat"))
print(len(files), "files found")

560 files found


In [3]:
#inspect the first 5 files
for file in files[:5]:
    print(file.name)


N09_M07_F10_K001_1.mat
N09_M07_F10_K001_10.mat
N09_M07_F10_K001_11.mat
N09_M07_F10_K001_12.mat
N09_M07_F10_K001_13.mat


In [4]:
#split the files names from its parts
recording_name = files[0].stem
parts = recording_name.split("_")
parts

['N09', 'M07', 'F10', 'K001', '1']

In [5]:
#extract bearning id and recording no
bearing_id  = parts[3]
recording_no = parts[4]

print("bearing id:", bearing_id)
print("recording no:", recording_no)

bearing id: K001
recording no: 1


In [6]:
#extract the operating condition code 
speed_code = parts[0]
torq_code = parts[1]
force_code = parts[2]
print("speed code:", speed_code)
print("torq code:", torq_code)
print("force code:", force_code)

speed code: N09
torq code: M07
force code: F10


In [7]:
record = {
    "bearing_id": bearing_id,
    "recording_no": recording_no,
    "speed_code": speed_code,
    "torq_code": torq_code,
    "force_code": force_code,
    "relative_path": files[0].relative_to(raw_folder).as_posix()
}

In [8]:
record

{'bearing_id': 'K001',
 'recording_no': '1',
 'speed_code': 'N09',
 'torq_code': 'M07',
 'force_code': 'F10',
 'relative_path': 'K001/K001/N09_M07_F10_K001_1.mat'}

In [9]:
#create registry of all recordings
records = []
for file in files:
    recording_name = file.stem
    parts = recording_name.split("_")
    bearing_id  = parts[3]
    recording_no = parts[4]
    speed_code = parts[0]
    torq_code = parts[1]
    force_code = parts[2]
    
    record = {
        "bearing_id": bearing_id,
        "recording_no": recording_no,
        "speed_code": speed_code,
        "torq_code": torq_code,
        "force_code": force_code,
        "relative_path": file.relative_to(raw_folder).as_posix()
    }
    
    records.append(record)
print("registry entries:", len(records))

registry entries: 560


In [10]:
#convert registry to a table
import pandas as pd
registry = pd.DataFrame(records)
print("rows:", registry.shape[0], "columns:", registry.shape[1])
registry.head()

rows: 560 columns: 6


,bearing_id,recording_no,speed_code,torq_code,force_code,relative_path
0,K001,1,N09,M07,F10,K001/K001/N09_M07_F10_K001_1.mat
1,K001,10,N09,M07,F10,K001/K001/N09_M07_F10_K001_10.mat
2,K001,11,N09,M07,F10,K001/K001/N09_M07_F10_K001_11.mat
3,K001,12,N09,M07,F10,K001/K001/N09_M07_F10_K001_12.mat
4,K001,13,N09,M07,F10,K001/K001/N09_M07_F10_K001_13.mat


In [11]:
registry['recording_id']=[Path(path).stem for path in registry["relative_path"]]

In [12]:
registry.head()

,bearing_id,recording_no,speed_code,torq_code,force_code,relative_path,recording_id
0,K001,1,N09,M07,F10,K001/K001/N09_M07_F10_K001_1.mat,N09_M07_F10_K001_1
1,K001,10,N09,M07,F10,K001/K001/N09_M07_F10_K001_10.mat,N09_M07_F10_K001_10
2,K001,11,N09,M07,F10,K001/K001/N09_M07_F10_K001_11.mat,N09_M07_F10_K001_11
3,K001,12,N09,M07,F10,K001/K001/N09_M07_F10_K001_12.mat,N09_M07_F10_K001_12
4,K001,13,N09,M07,F10,K001/K001/N09_M07_F10_K001_13.mat,N09_M07_F10_K001_13


In [13]:
#check to duplicate recording ids
duplicate_recording_ids = registry['recording_id'].duplicated()
print("duplicate recording ids:", duplicate_recording_ids.sum())


duplicate recording ids: 0


In [14]:
#count recordings of each bearing
registry['bearing_id'].value_counts()

bearing_id
K001    80
K002    80
K003    80
KA04    80
KA15    80
KI04    80
KI14    80
Name: count, dtype: int64

In [15]:
#counts recording of each operating condition
condition_counts = registry.groupby(['speed_code', 'torq_code', 'force_code']).size()
print(condition_counts)

speed_code  torq_code  force_code
N09         M07        F10           140
N15         M01        F10           140
            M07        F04           140
                       F10           140
dtype: int64


In [16]:
registry = registry.sort_values(by=['bearing_id', 'speed_code', 'torq_code', 'force_code', 'recording_no']).reset_index(drop=True)
registry.head()

,bearing_id,recording_no,speed_code,torq_code,force_code,relative_path,recording_id
0,K001,1,N09,M07,F10,K001/K001/N09_M07_F10_K001_1.mat,N09_M07_F10_K001_1
1,K001,10,N09,M07,F10,K001/K001/N09_M07_F10_K001_10.mat,N09_M07_F10_K001_10
2,K001,11,N09,M07,F10,K001/K001/N09_M07_F10_K001_11.mat,N09_M07_F10_K001_11
3,K001,12,N09,M07,F10,K001/K001/N09_M07_F10_K001_12.mat,N09_M07_F10_K001_12
4,K001,13,N09,M07,F10,K001/K001/N09_M07_F10_K001_13.mat,N09_M07_F10_K001_13


In [17]:
registry["recording_no"] = registry["recording_no"].astype(int)

registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

print(registry[["recording_no", "recording_id"]].head())

   recording_no        recording_id
0             1  N09_M07_F10_K001_1
1             2  N09_M07_F10_K001_2
2             3  N09_M07_F10_K001_3
3             4  N09_M07_F10_K001_4
4             5  N09_M07_F10_K001_5


In [18]:
# Use the complete fixed plan, even when rerunning earlier learning cells.
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

partition    condition
development  faulty       160
             healthy      160
final_test   faulty       160
             healthy       80
dtype: int64


In [19]:
output_path = raw_folder.parents[2] / "metadata" / "recording_registry.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
print("Registry destination:", output_path)
print("The complete registry will be saved in the final cell.")

Registry destination: C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection\metadata\recording_registry.csv
The complete registry will be saved in the final cell.


In [20]:
# Preview the current table; this is not a disk-readback check.
saved_registry = registry.copy()
print("Working rows and columns:", saved_registry.shape)
print(saved_registry.groupby(["bearing_id", "condition", "partition"]).size())

Working rows and columns: (560, 9)
bearing_id  condition  partition  
K001        healthy    development    80
K002        healthy    development    80
K003        healthy    final_test     80
KA04        faulty     development    80
KA15        faulty     final_test     80
KI04        faulty     development    80
KI14        faulty     final_test     80
dtype: int64


In [21]:
k002_files = list(raw_folder.rglob("*_K002_*.mat"))

print("K002 recordings found:", len(k002_files))

K002 recordings found: 80


In [22]:
files = sorted(raw_folder.rglob("*.mat"))

print("Total recordings:", len(files))

Total recordings: 560


In [23]:
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print("Rows and columns:", registry.shape)
print(registry["bearing_id"].value_counts())

Rows and columns: (560, 7)
bearing_id
K001    80
K002    80
K003    80
KA04    80
KA15    80
KI04    80
KI14    80
Name: count, dtype: int64


In [24]:
# Use the complete fixed plan, even when rerunning earlier learning cells.
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

partition    condition
development  faulty       160
             healthy      160
final_test   faulty       160
             healthy       80
dtype: int64


In [25]:
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())
print("Missing values:", registry.isna().sum().sum())

Duplicate IDs: 0
Missing values: 0


In [26]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

# Save only after the audit joins at the end of the notebook.

print("Working rows:", len(registry))
print("Location:", output_path)

Working rows: 560
Location: C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection\metadata\recording_registry.csv


In [27]:
ka04_files = list(raw_folder.rglob("*_KA04_*.mat"))

print("KA04 recordings found:", len(ka04_files))


KA04 recordings found: 80


In [28]:
files = sorted(raw_folder.rglob("*.mat"))

print("Total recordings:", len(files))

Total recordings: 560


In [29]:
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print(registry["bearing_id"].value_counts())

bearing_id
K001    80
K002    80
K003    80
KA04    80
KA15    80
KI04    80
KI14    80
Name: count, dtype: int64


In [30]:
# Use the complete fixed plan, even when rerunning earlier learning cells.
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

partition    condition
development  faulty       160
             healthy      160
final_test   faulty       160
             healthy       80
dtype: int64


In [31]:
print("Rows and columns:", registry.shape)
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())
print("Missing values:", registry.isna().sum().sum())

Rows and columns: (560, 9)
Duplicate IDs: 0
Missing values: 0


In [32]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

# Save only after the audit joins at the end of the notebook.

print("Working rows:", len(registry))

Working rows: 560


In [33]:
ki04_files = list(raw_folder.rglob("*_KI04_*.mat"))

print("KI04 recordings found:", len(ki04_files))

KI04 recordings found: 80


In [34]:
files = sorted(raw_folder.rglob("*.mat"))
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print("Total recordings:", len(registry))
print(registry["bearing_id"].value_counts())

Total recordings: 560
bearing_id
K001    80
K002    80
K003    80
KA04    80
KA15    80
KI04    80
KI14    80
Name: count, dtype: int64


In [35]:
# Use the complete fixed plan, even when rerunning earlier learning cells.
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

partition    condition
development  faulty       160
             healthy      160
final_test   faulty       160
             healthy       80
dtype: int64


In [36]:
print("Rows and columns:", registry.shape)
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())
print("Missing values:", registry.isna().sum().sum())

Rows and columns: (560, 9)
Duplicate IDs: 0
Missing values: 0


In [37]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

# Save only after the audit joins at the end of the notebook.

print("Working rows:", len(registry))

Working rows: 560


In [38]:
k003_files = list(raw_folder.rglob("*_K003_*.mat"))

print("K003 recordings found:", len(k003_files))

K003 recordings found: 80


In [39]:
ka15_files = list(raw_folder.rglob("*_KA15_*.mat"))

print("KA15 recordings found:", len(ka15_files))

KA15 recordings found: 80


In [40]:
ki14_files = list(raw_folder.rglob("*_KI14_*.mat"))

print("KI14 recordings found:", len(ki14_files))

KI14 recordings found: 80


In [41]:
files = sorted(raw_folder.rglob("*.mat"))
records = []

for file in files:
    parts = file.stem.split("_")

    records.append({
        "recording_id": file.stem,
        "bearing_id": parts[3],
        "recording_no": int(parts[4]),
        "speed_code": parts[0],
        "torq_code": parts[1],
        "force_code": parts[2],
        "relative_path": file.relative_to(raw_folder).as_posix()
    })

registry = pd.DataFrame(records)

print("Total recordings:", len(registry))
print(registry["bearing_id"].value_counts().sort_index())

Total recordings: 560
bearing_id
K001    80
K002    80
K003    80
KA04    80
KA15    80
KI04    80
KI14    80
Name: count, dtype: int64


In [42]:
registry["condition"] = registry["bearing_id"].map({
    "K001": "healthy",
    "K002": "healthy",
    "K003": "healthy",
    "KA04": "faulty",
    "KA15": "faulty",
    "KI04": "faulty",
    "KI14": "faulty"
})

registry["partition"] = registry["bearing_id"].map({
    "K001": "development",
    "K002": "development",
    "KA04": "development",
    "KI04": "development",
    "K003": "final_test",
    "KA15": "final_test",
    "KI14": "final_test"
})

print(registry.groupby(["partition", "condition"]).size())

partition    condition
development  faulty       160
             healthy      160
final_test   faulty       160
             healthy       80
dtype: int64


In [43]:
print("Missing values:", registry.isna().sum().sum())
print("Duplicate IDs:", registry["recording_id"].duplicated().sum())

partitions_per_bearing = registry.groupby("bearing_id")["partition"].nunique()
print("Partitions per bearing:")
print(partitions_per_bearing)

Missing values: 0
Duplicate IDs: 0
Partitions per bearing:
bearing_id
K001    1
K002    1
K003    1
KA04    1
KA15    1
KI04    1
KI14    1
Name: partition, dtype: int64


In [44]:
registry = registry.sort_values(
    ["bearing_id", "speed_code", "torq_code", "force_code", "recording_no"]
).reset_index(drop=True)

# Save only after the audit joins at the end of the notebook.

print("Working rows:", len(registry))

Working rows: 560


In [45]:
registry.shape

(560, 9)

In [46]:
from pathlib import Path
import pandas as pd

project_root = Path(
    r"C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection"
)
raw_folder = project_root / "data" / "raw" / "paderborn"
registry_path = project_root / "metadata" / "recording_registry.csv"

# Reload a saved registry when available; otherwise use the inventory above.
if registry_path.is_file():
    registry = pd.read_csv(registry_path)

# Start with base columns so later merges do not duplicate audit columns.
registry = registry[[
    "recording_id",
    "bearing_id",
    "recording_no",
    "speed_code",
    "torq_code",
    "force_code",
    "relative_path",
    "condition",
    "partition"
]].copy()

if registry["recording_id"].duplicated().any() or registry.isna().any().any():
    raise ValueError("Check duplicate IDs and missing base metadata.")
if not registry.groupby("bearing_id")["partition"].nunique().eq(1).all():
    raise ValueError("A bearing crosses partition boundaries.")

print("Base rows and columns:", registry.shape)
file_exists = [
    (raw_folder / path).is_file()
    for path in registry["relative_path"]
]
print("Files found:", sum(file_exists))
print("Files missing:", len(file_exists) - sum(file_exists))
if not all(file_exists):
    raise FileNotFoundError("A recording listed in the registry is missing.")

Base rows and columns: (560, 9)
Files found: 560
Files missing: 0


In [47]:
from scipy.io import loadmat
row = registry[registry["partition"] == "development"].iloc[0]
file_path = raw_folder / row["relative_path"]

recording = loadmat(file_path)[file_path.stem][0, 0]
channels = recording["Y"].ravel()
names = [channel["Name"].item() for channel in channels]

print("Recording:", row["recording_id"])
print("Channels:", names)

Recording: N09_M07_F10_K001_1
Channels: ['force', 'phase_current_1', 'phase_current_2', 'speed', 'temp_2_bearing_module', 'torque', 'vibration_1']


In [48]:
vibration = channels[names.index("vibration_1")]
x = vibration["Data"].ravel()

axis_index = int(vibration["XIndex"].item()) - 1
axis = recording["X"].ravel()[axis_index]
t_stored = axis["Data"].ravel()

print("Vibration samples:", x.size)
print("Time values:", t_stored.size)
print("Channel raster:", vibration["Raster"].item())
print("Axis raster:", axis["Raster"].item())
print("Stored unit:", vibration["Unit"])

Vibration samples: 256823
Time values: 256823
Channel raster: HostService
Axis raster: HostService
Stored unit: []


In [49]:
time_span = t_stored[-1] - t_stored[0]
fs_check = (x.size - 1) / time_span

print(f"Stored time span: {time_span:.6f} seconds")
print(f"Rate from time span: {fs_check:.4f} Hz")

Stored time span: 4.000003 seconds
Rate from time span: 64205.4451 Hz


In [50]:
import numpy as np

dt = np.diff(t_stored)

print("Time strictly increasing:", (dt > 0).all())
print("Mean interval (microseconds):", dt.mean() * 1e6)
print("Median interval (microseconds):", np.median(dt) * 1e6)
print("Smallest interval (microseconds):", dt.min() * 1e6)
print("Largest interval (microseconds):", dt.max() * 1e6)

Time strictly increasing: True
Mean interval (microseconds): 15.57500300515315
Median interval (microseconds): 15.398052489068448
Smallest interval (microseconds): 4.277236802074924
Largest interval (microseconds): 15.825776169542394


In [51]:
t_nominal = np.arange(x.size) / 64_000
timing_error = (t_stored - t_stored[0]) - t_nominal

print("Maximum deviation (ms):", np.abs(timing_error).max() * 1000)
print("Final deviation (ms):", timing_error[-1] * 1000)

Maximum deviation (ms): 12.840328210558027
Final deviation (ms): -12.840328210558027


In [52]:
comparison_path = (
    raw_folder / "K001" / "K001" / "N15_M07_F10_K001_1.mat"
)

comparison = loadmat(comparison_path)[comparison_path.stem][0, 0]
comparison_channels = comparison["Y"].ravel()

comparison_names = [
    channel["Name"].item() for channel in comparison_channels
]
comparison_vibration = comparison_channels[
    comparison_names.index("vibration_1")
]

comparison_x = comparison_vibration["Data"].ravel()
axis_index = int(comparison_vibration["XIndex"].item()) - 1
comparison_time = comparison["X"].ravel()[axis_index]["Data"].ravel()

span = comparison_time[-1] - comparison_time[0]

print("Comparison samples:", comparison_x.size)
print("Comparison span (s):", span)
print("Comparison axis-derived rate (Hz):", (comparison_x.size - 1) / span)

Comparison samples: 256001
Comparison span (s): 3.999998716828959
Comparison axis-derived rate (Hz): 64000.02053074324


In [53]:
def inspect_timing(path):
    recording = loadmat(path)[path.stem][0, 0]
    channels = recording["Y"].ravel()
    names = [channel["Name"].item() for channel in channels]

    vibration = channels[names.index("vibration_1")]
    samples = vibration["Data"].size

    axis_index = int(vibration["XIndex"].item()) - 1
    time = recording["X"].ravel()[axis_index]["Data"].ravel()
    span = time[-1] - time[0]

    return {
        "recording_id": path.stem,
        "sample_count": samples,
        "time_count": time.size,
        "axis_span_s": span,
        "axis_rate_hz": (time.size - 1) / span
    }

In [54]:
timing_result = inspect_timing(comparison_path)

print(timing_result)

{'recording_id': 'N15_M07_F10_K001_1', 'sample_count': 256001, 'time_count': 256001, 'axis_span_s': np.float64(3.999998716828959), 'axis_rate_hz': np.float64(64000.02053074324)}


In [55]:
timing_records = []

for relative_path in registry["relative_path"]:
    path = raw_folder / relative_path
    timing_records.append(inspect_timing(path))

timing_table = pd.DataFrame(timing_records)

print("Recordings inspected:", len(timing_table))
timing_table.head()

Recordings inspected: 560


,recording_id,sample_count,time_count,axis_span_s,axis_rate_hz
0,N09_M07_F10_K001_1,256823,256823,4.000003,64205.445076
1,N09_M07_F10_K001_2,256001,256001,4.000000,64000.006844
2,N09_M07_F10_K001_3,256001,256001,4.000000,64000.006844
3,N09_M07_F10_K001_4,256001,256001,3.999995,64000.075279
4,N09_M07_F10_K001_5,256001,256001,3.999999,64000.013687


In [56]:
timing_table["axis_rate_difference_pct"] = (
    (timing_table["axis_rate_hz"] - 64_000) / 64_000
).abs() * 100

timing_table.sort_values(
    "axis_rate_difference_pct", ascending=False
).head(10)

,recording_id,sample_count,time_count,axis_span_s,axis_rate_hz,axis_rate_difference_pct
109,N15_M01_F10_K002_10,291847,291847,3.999997,72961.546811,14.002417
157,N15_M07_F10_K002_18,278265,278265,3.999997,69566.044632,8.696945
365,N15_M07_F04_KA15_6,278139,278139,4.000000,69534.507435,8.647668
559,N15_M07_F10_KI14_20,274159,274159,3.999998,68539.529316,7.093015
522,N15_M07_F04_KI14_3,272879,272879,3.999997,68219.551063,6.593049
557,N15_M07_F10_KI14_18,265780,265780,3.999993,66444.863680,3.820100
275,N15_M01_F10_KA04_16,262735,262735,3.999991,65683.647497,2.630699
180,N15_M01_F10_K003_1,262593,262593,3.999987,65648.210598,2.575329
203,N15_M07_F04_K003_4,262512,262512,3.999989,65627.932462,2.543644
209,N15_M07_F04_K003_10,262243,262243,3.999997,65560.549074,2.438358


In [57]:
review_threshold_pct = 0.1

timing_table["needs_timing_review"] = (
    timing_table["axis_rate_difference_pct"] > review_threshold_pct
)

print("Recordings checked:", len(timing_table))
print("Flagged for timing review:", timing_table["needs_timing_review"].sum())

Recordings checked: 560
Flagged for timing review: 79


In [58]:
review_id = "N15_M01_F10_K002_10"

review_path = raw_folder / registry.loc[
    registry["recording_id"] == review_id, "relative_path"
].iloc[0]

review = loadmat(review_path)[review_path.stem][0, 0]
review_channels = review["Y"].ravel()
review_names = [channel["Name"].item() for channel in review_channels]
review_vibration = review_channels[review_names.index("vibration_1")]

axis_index = int(review_vibration["XIndex"].item()) - 1
review_axis = review["X"].ravel()[axis_index]
review_time = review_axis["Data"].ravel()
dt = np.diff(review_time)

print("Channel raster:", review_vibration["Raster"].item())
print("Axis raster:", review_axis["Raster"].item())
print("DownSampling:", review_vibration["DownSampling"].item())
print("Time values finite:", np.isfinite(review_time).all())
print("Time strictly increasing:", (dt > 0).all())
print("Interval percentiles (µs):",
      np.percentile(dt, [0, 25, 50, 75, 100]) * 1e6)

Channel raster: HostService
Axis raster: HostService
DownSampling: 1
Time values finite: True
Time strictly increasing: True
Interval percentiles (µs): [ 4.27721851 15.39798663 15.39798663 15.82570848 43.62762878]


In [59]:
timing_table["timing_status"] = np.where(
    timing_table["needs_timing_review"],
    "needs_review",
    "within_span_tolerance"
)

audit_path = project_root / "metadata" / "timing_audit.csv"
timing_table.to_csv(audit_path, index=False)

print(timing_table["timing_status"].value_counts())

timing_status
within_span_tolerance    481
needs_review              79
Name: count, dtype: int64


In [60]:
length_mismatch = (
    timing_table["sample_count"] != timing_table["time_count"]
)

print("Recordings with mismatched lengths:", length_mismatch.sum())

Recordings with mismatched lengths: 0


In [61]:
def inspect_quality(path):
    recording = loadmat(path)[path.stem][0, 0]
    channels = recording["Y"].ravel()
    names = [channel["Name"].item() for channel in channels]

    vibration = channels[names.index("vibration_1")]
    x = vibration["Data"].ravel()

    axis_index = int(vibration["XIndex"].item()) - 1
    axis = recording["X"].ravel()[axis_index]
    time = axis["Data"].ravel()

    return {
        "recording_id": path.stem,
        "signal_finite": np.isfinite(x).all(),
        "time_finite": np.isfinite(time).all(),
        "time_increasing": (np.diff(time) > 0).all(),
        "rasters_match": vibration["Raster"].item() == axis["Raster"].item()
    }

In [62]:
quality_result = inspect_quality(review_path)

print(quality_result)

{'recording_id': 'N15_M01_F10_K002_10', 'signal_finite': np.True_, 'time_finite': np.True_, 'time_increasing': np.True_, 'rasters_match': True}


In [63]:
quality_records = []

for relative_path in registry["relative_path"]:
    path = raw_folder / relative_path
    quality_records.append(inspect_quality(path))

quality_table = pd.DataFrame(quality_records)

print("Recordings checked:", len(quality_table))
print(quality_table.drop(columns="recording_id").sum())

Recordings checked: 560
signal_finite      560
time_finite        560
time_increasing    560
rasters_match      560
dtype: int64


In [64]:
quality_path = project_root / "metadata" / "quality_audit.csv"

quality_table.to_csv(quality_path, index=False)

print("Saved:", quality_path)

Saved: C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection\metadata\quality_audit.csv


In [65]:
registry["speed_setting_rpm"] = registry["speed_code"].map({
    "N09": 900, "N15": 1500
})

registry["torque_setting_nm"] = registry["torq_code"].map({
    "M01": 0.1, "M07": 0.7
})

registry["radial_force_setting_n"] = registry["force_code"].map({
    "F04": 400, "F10": 1000
})

registry[[
    "speed_setting_rpm",
    "torque_setting_nm",
    "radial_force_setting_n"
]].drop_duplicates()

,speed_setting_rpm,torque_setting_nm,radial_force_setting_n
0,900,0.7,1000
20,1500,0.1,1000
40,1500,0.7,400
60,1500,0.7,1000


In [66]:
registry["vibration_channel"] = "vibration_1"
registry["nominal_fs_hz"] = 64_000
registry["vibration_unit_status"] = "unverified"

registry[[
    "vibration_channel",
    "nominal_fs_hz",
    "vibration_unit_status"
]].drop_duplicates()

,vibration_channel,nominal_fs_hz,vibration_unit_status
0,vibration_1,64000,unverified


In [67]:
checked_registry = registry.merge(
    timing_table,
    on="recording_id",
    how="left",
    validate="one_to_one"
)

checked_registry = checked_registry.merge(
    quality_table,
    on="recording_id",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(checked_registry))
print(checked_registry["timing_status"].value_counts())

Rows: 560
timing_status
within_span_tolerance    481
needs_review              79
Name: count, dtype: int64


In [68]:
print("Rows:", len(checked_registry))
print("Missing values:", checked_registry.isna().sum().sum())
print("Duplicate IDs:", checked_registry["recording_id"].duplicated().sum())

Rows: 560
Missing values: 0
Duplicate IDs: 0


In [69]:
# Stop before writing an incomplete or duplicated registry.
if len(checked_registry) != len(registry):
    raise ValueError("The audit joins changed the recording count.")
if checked_registry.isna().any().any():
    raise ValueError("Missing metadata or audit results: registry was not saved.")
if checked_registry["recording_id"].duplicated().any():
    raise ValueError("Duplicate IDs: registry was not saved.")

registry_path = project_root / "metadata" / "recording_registry.csv"
registry_path.parent.mkdir(parents=True, exist_ok=True)
checked_registry.to_csv(registry_path, index=False)

print("Saved recordings:", len(checked_registry))
print("Location:", registry_path)

Saved recordings: 560
Location: C:\Users\redhu\OneDrive\Desktop\Bearing_fault_detection\metadata\recording_registry.csv


In [70]:
review_development = checked_registry[
    (checked_registry["partition"] == "development")
    & checked_registry["needs_timing_review"]
].copy()

print("Flagged development recordings:", len(review_development))

print(review_development.groupby(
    ["bearing_id", "speed_code", "torq_code", "force_code"]
).size())

Flagged development recordings: 48
bearing_id  speed_code  torq_code  force_code
K001        N09         M07        F10            4
            N15         M01        F10            4
                        M07        F04           10
                                   F10            4
K002        N09         M07        F10            2
            N15         M01        F10            2
                        M07        F10            1
KA04        N09         M07        F10            1
            N15         M01        F10            3
                        M07        F04            2
                                   F10            2
KI04        N09         M07        F10            4
            N15         M01        F10            1
                        M07        F04            5
                                   F10            3
dtype: int64


In [71]:
same_setting = checked_registry[
    (checked_registry["bearing_id"] == "K002")
    & (checked_registry["speed_code"] == "N15")
    & (checked_registry["torq_code"] == "M01")
    & (checked_registry["force_code"] == "F10")
]

same_setting.sort_values("recording_no")[[
    "recording_id",
    "sample_count",
    "axis_span_s",
    "axis_rate_hz",
    "needs_timing_review"
]]

,recording_id,sample_count,axis_span_s,axis_rate_hz,needs_timing_review
100,N15_M01_F10_K002_1,261265,3.999999,65316.020953,True
101,N15_M01_F10_K002_2,256001,3.999992,64000.130028,False
102,N15_M01_F10_K002_3,256001,4.000000,64000.000000,False
103,N15_M01_F10_K002_4,256001,4.000000,64000.006844,False
104,N15_M01_F10_K002_5,256003,3.999999,64000.520531,False
105,N15_M01_F10_K002_6,256001,4.000000,64000.000000,False
106,N15_M01_F10_K002_7,256001,4.000000,64000.006844,False
107,N15_M01_F10_K002_8,256001,4.000000,64000.000000,False
108,N15_M01_F10_K002_9,256003,3.999994,64000.588967,False
109,N15_M01_F10_K002_10,291847,3.999997,72961.546811,True


In [72]:
print("Recording:", review_path.stem)

for channel in review["Y"].ravel():
    print(
        channel["Name"].item(),
        "— samples:", channel["Data"].size,
        "— raster:", channel["Raster"].item()
    )

Recording: N15_M01_F10_K002_10
force — samples: 16001 — raster: Mech_4kHz
phase_current_1 — samples: 291847 — raster: HostService
phase_current_2 — samples: 291847 — raster: HostService
speed — samples: 16001 — raster: Mech_4kHz
temp_2_bearing_module — samples: 5 — raster: Temp_1Hz
torque — samples: 16001 — raster: Mech_4kHz
vibration_1 — samples: 291847 — raster: HostService


In [73]:
description = review["Description"][0, 0]

for name in description.dtype.names:
    details = description[name]
    print(name, "— fields:", details.dtype.names)

General — fields: ('User', 'DateTime', 'Origin', 'Description')
Recording — fields: ('StartCondition', 'StopCondition')
Measurement — fields: ('XAxisOffset', 'Length', 'StartTimestamp', 'StopTimestamp')


In [74]:
measurement = description["Measurement"][0, 0]

for name in measurement.dtype.names:
    print(name, ":", measurement[name])

XAxisOffset : [[0]]
Length : [[4]]
StartTimestamp : [[-0.00263477]]
StopTimestamp : [[3.99736523]]


In [75]:
general = description["General"][0, 0]

print("Origin:", general["Origin"])
print("Description:", general["Description"])

Origin: ["Software revision: modell_autass_05_14.sdf, modell_autass_05_14.x86\r\nVariable description: modell_autass_05_14.sdf\r\n    Platform / Device: Platform [DS1006 Processor Board, assigned to 'ds1006_5']"]
Description: []


In [76]:
analysis_development = checked_registry[
    (checked_registry["partition"] == "development")
    & (~checked_registry["needs_timing_review"])
].copy()

print("Provisional development recordings:", len(analysis_development))
print(analysis_development.groupby(["bearing_id", "condition"]).size())

Provisional development recordings: 272
bearing_id  condition
K001        healthy      58
K002        healthy      75
KA04        faulty       72
KI04        faulty       67
dtype: int64
